# Tarea 8 · MERGE incremental de mediciones

Ejecutar después de reconstruir las tablas Silver intermedias con `01_transformar_silver.ipynb`. Este notebook no modifica Bronze ni sus checkpoints.


In [1]:
import json
import sys
from pathlib import Path
from pyspark.sql import functions as F

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config' / 'silver.json').is_file()), None)
if project_root is None:
    raise FileNotFoundError('No se encuentra config/silver.json')
sys.path.insert(0, str(project_root / 'scripts'))
from merge_mediciones import procesar_incremental, historial_merge
silver = json.loads((project_root / 'config' / 'silver.json').read_text(encoding='utf-8'))
bronze = json.loads((project_root / silver['bronze_config']).read_text(encoding='utf-8'))
tables = silver['tables']
print('Origen:', bronze['tables']['mediciones'])
print('Intermedia:', tables['mediciones'])
print('Definitiva:', tables['mediciones_vigentes'])


Origen: workspace.lensdata_bronze.mediciones
Intermedia: workspace.lensdata_silver.mediciones
Definitiva: workspace.lensdata_silver.mediciones_vigentes


## Ingesta incremental

La cobertura por archivo se valida antes del MERGE. Solo se procesan archivos no confirmados en el registro persistente.


In [2]:
resultado = procesar_incremental(spark, bronze, silver)
print(resultado)


{'candidatas': 467, 'reenvios_en_lote': 1, 'conflictos_en_lote': 0, 'versiones_superadas_en_lote': 2, 'conflictos_con_destino': 0, 'archivos': 15}


## Validación posterior

Las métricas de `operationMetrics` dependen del runtime Delta. Las cifras observadas deben anotarse tras ejecutar el notebook.


In [3]:
target = spark.table(tables['mediciones_vigentes'])
key = silver['entities']['mediciones']['business_key']
assert target.count() == target.select(*key).distinct().count()
assert target.filter(F.col('_source_file').isNull()).count() == 0
assert target.filter(F.col('version_origen') < 1).count() == 0
display(spark.sql(f"DESCRIBE DETAIL {tables['mediciones_vigentes']}").select('format', 'numFiles'))
display(target.groupBy('version_origen').count().orderBy('version_origen'))
display(spark.table(tables['archivos_procesados']).orderBy('_source_file'))
display(spark.table(tables['conflictos_version']).orderBy('processed_at'))
display(historial_merge(spark, tables['mediciones_vigentes']))


,format,numFiles
0,delta,1


,version_origen,count
0,1,466
1,2,1


,_source_file,bronze_row_count,processed_at
0,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_01/mediciones_2026-11-01.csv,40,2026-11-26 15:14:52.432108
1,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_02/mediciones_2026-11-02.csv,40,2026-11-26 15:14:52.432108
2,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-03.csv,39,2026-11-26 15:14:52.432108
3,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-04.csv,39,2026-11-26 15:14:52.432108
4,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-05.csv,39,2026-11-26 15:14:52.432108
5,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-06.csv,39,2026-11-26 15:14:52.432108
6,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-07.csv,39,2026-11-26 15:14:52.432108
7,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-08.csv,39,2026-11-26 15:14:52.432108
8,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-09.csv,39,2026-11-26 15:14:52.432108
9,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-10.csv,39,2026-11-26 15:14:52.432108


,conflict_id,conflict_code,plataforma,video_id,fecha_medicion,version_origen,source_file,source_metrics,target_source_file,target_metrics,processed_at


,version,timestamp,operation,operationMetrics
0,1,2026-11-26 15:14:39.985,MERGE,"{'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '0', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '0', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '0', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'rewriteTimeMs': '1412', 'numTargetFilesAdded': '1', 'numTargetBytesAdded': '7434', 'executionTimeMs': '5535', 'materializeSourceTimeMs': '1782', 'numTargetRowsInserted': '467', 'numTargetDeletionVectorsUpdated': '0', 'scanTimeMs': '2215', 'numOutputRows': '467', 'numTargetDeletionVectorsRemoved': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numSourceRows': '467', 'numTargetFilesRemoved': '0'}"
1,0,2026-11-26 15:14:25.335,CREATE TABLE,{}


## Reprocesamiento controlado

Indicar rutas completas `_source_file` de Bronze solo cuando sea necesario repetir archivos concretos. El checkpoint Bronze no se modifica.


In [4]:
archivos_reprocesar = []  # Ejemplo: ['/Volumes/.../lote_correccion/correccion_dia_1.csv']
if archivos_reprocesar:
    print(procesar_incremental(spark, bronze, silver, reprocesar_archivos=archivos_reprocesar))
    display(historial_merge(spark, tables['mediciones_vigentes'], limit=3))
